# transistor：未知の回転角度に対する異常検知性能

検証方針①です。**transistorのみ／保存済みSSE版Hybrid 2モデル／再学習なし**。
36度刻みで学習したモデルが、その間の角度にも対応するかを調べます。

| 条件 | 角度 | 評価方法 |
| --- | --- | --- |
| 回転なし | 0度 | 前回の保存済みスコアを読み込む。再推論しない |
| 既知角度 | 36〜324度の36度刻み | 前回の単一割り当て100枚の保存済みスコアを読む |
| 中間角度 | 18・54・90・…・342度の10候補 | seed42〜46の各回、画像ごとに1角度を独立・等確率で抽選 |
| 未知ランダム角度 | 0〜359度の整数のうち36度の倍数を除く350候補 | 同じ5シードで、各回1画像1角度を抽選 |

- 各回は同じ正常60枚・異常40枚の**100枚**。両モデルに全く同じ回転画像を入力します。
- 新規入力は2条件×5回×100枚＝延べ1,000枚、2モデルで2,000スコア。独立した1,000画像ではありません。
- 性能指標はAUROCのみ。閾値探索・モデル選択・追加学習はしません。
- 5回の標準偏差は**角度割り当てによる変動**で、学習シード間の変動や信頼区間ではありません。
- 未知ランダム角度には中間角度も含まれます。2条件は排他的な集合ではありません。連続角度ではなく1度単位の評価です。
- 新規学習用の線形βアニーリングは今回適用しません。保存済みモデルをそのまま使用します。

**実行はユーザーが行います。作成時点では未実行です。** 上から順に実行してください。
評価セルを再実行すると、既存結果を上書きせず新しい実行フォルダを作成します。
入出力・誤差マップも圧縮保存します（10ケース合計で非圧縮換算約330 MB、実際の圧縮後容量は画像に依存）。
可視化の枚数・条件・シードを後で変更しても、保存した評価時の配列から表示できます。


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
from io import BytesIO
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib
import importlib.metadata
import json
import math
import platform
import random
import uuid
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torchvision import transforms
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display
from skimage.metrics import structural_similarity as ssim

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p/'memory/memory.md').is_file() and (p/'data/transistor').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('このリポジトリ内を作業ディレクトリにしてください。')
NORMAL_RUN = ROOT/'saved_models/transistor_update_matched_20261106/20261106_021112_b266ecab'
ROTATION_RUN = ROOT/'saved_models/transistor_rotation_control_20261106_reflect/20261106_002435_9b4da165'
BASELINE_DIR = NORMAL_RUN/'evaluations/20261106_025126_58aad9c8'
KNOWN_DIR = ROOT/'saved_models/transistor_rotated_test_20261106/20261106_031022_6bed84fa'
OUTPUT_BASE = ROOT/'saved_models/transistor_unseen_rotation_20261117'
MODEL_PATHS = {'A_original':NORMAL_RUN/'model_final.pth',
               'B_rotate10':ROTATION_RUN/'seed_42/B_rotate10/model_final.pth'}
EXPECTED_HASHES = {'A_original':'380bb28d611612ecda31f97dee299c2310e1b39f6c52b2fbd92144e2ab3e4667',
                   'B_rotate10':'20b0568e0cbae99290cb474b7e327a95983368fb37ece7edbcd0cd1c91f42de5'}
CONDITIONS = tuple(MODEL_PATHS)
LABELS = {'A_original':'A：通常学習','B_rotate10':'B：回転拡張'}
COLORS = {'A_original':'#526b8e','B_rotate10':'darkorange'}
SCENARIO_LABELS = {'unrotated':'回転なし（保存済み）',
    'known_rotated':'既知角度（保存済み）',
    'midpoint':'中間角度（5回）','unseen_random':'未知ランダム角度（5回）'}
CATEGORY_LABELS = {'good':'正常','bent_lead':'端子の曲がり','cut_lead':'端子の切断',
                   'damaged_case':'本体の損傷','misplaced':'配置ずれ'}
TRAINING_ANGLES = tuple(range(0,360,36))
ANGLE_CANDIDATES = {
    'midpoint': tuple(range(18,360,36)),
    'unseen_random': tuple(a for a in range(360) if a % 36 != 0),
}
ASSIGNMENT_SEEDS = (42,43,44,45,46)
ASSIGNMENT_SEED = 42  # 推論環境の乱数固定用。評価は平均μから行う。
REPRESENTATIVE_SEED = 42  # ROC・ヒストグラム用。結果を見て選び直さない。
VISUAL_SCENARIOS = ('midpoint','unseen_random')
VISUAL_SEEDS = (42,)  # 別の回も例：(42,43,44,45,46)
DEVICE = torch.device('cpu')  # 既存の評価と同じ環境
EVAL_BATCH_SIZE = 32
TOP_K = 130
# 以下は可視化だけの設定。評価は全条件・全5回・各100枚です。
VISUAL_PATHS = None  # 例：['good/000.png','cut_lead/000.png']。Noneなら種類ごとの先頭画像
MAX_IMAGES_PER_CATEGORY = 1  # VISUAL_PATHS=Noneのとき。Noneなら全画像

random.seed(ASSIGNMENT_SEED)
np.random.seed(ASSIGNMENT_SEED)
torch.manual_seed(ASSIGNMENT_SEED)
torch.set_num_threads(min(4,max(1,torch.get_num_threads())))
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
for font in ['Yu Gothic','Meiryo','MS Gothic','Noto Sans CJK JP','IPAexGothic']:
    if font in available_fonts:
        plt.rcParams['font.family'] = font
        break
else:
    import japanize_matplotlib
plt.rcParams.update({'font.size':11,'axes.unicode_minus':False,'figure.dpi':100})
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))
def write_json(path,value):
    path.write_text(json.dumps(value,ensure_ascii=False,indent=2)+chr(10),encoding='utf-8')
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


## 1. 保存済みモデルと前回結果の照合

通常学習は `20261106_021112_b266ecab`、回転拡張は `20261106_002435_9b4da165` のHybridを使用。
どちらもseed42・6,700更新のSSE版です。MSE統合検証のモデルではありません。
重み・設定・画像の識別情報を照合し、異なる条件のファイルを誤って混ぜないようにします。


In [ ]:
normal_config = read_json(NORMAL_RUN/'config.json')
rotation_config = read_json(ROTATION_RUN/'config.json')
normal_summary = read_json(NORMAL_RUN/'summary.json')
rotation_summary = read_json(MODEL_PATHS['B_rotate10'].parent/'summary.json')
assert normal_config['status'] == rotation_config['status'] == 'completed'
assert normal_summary['updates'] == rotation_summary['updates'] == 6700
assert normal_config['seed'] == normal_summary['seed'] == rotation_summary['seed'] == 42
assert normal_summary['initial_state_sha256'] == rotation_summary['initial_state_sha256']
assert normal_config['source_model_sha256'] == EXPECTED_HASHES['B_rotate10']
assert normal_config['model'] == 'Hybrid CNN + FC'
assert rotation_config['angles'] == list(TRAINING_ANGLES)
assert rotation_config['rotation_background'] == 'reflect'
assert rotation_config['rotation_interpolation'] == 'NEAREST'
assert rotation_config['rotation_expand'] is False
assert rotation_config['rotation_stage'] == 'full_resolution_before_resize'
assert rotation_config['source_size'] == [1024,1024]
assert normal_config['latent_dim'] == rotation_config['latent_dim'] == 64
assert normal_summary['model_sha256'] == EXPECTED_HASHES['A_original']
assert all(sha256(path) == EXPECTED_HASHES[name] for name,path in MODEL_PATHS.items())
source_files = [*MODEL_PATHS.values(), NORMAL_RUN/'config.json', ROTATION_RUN/'config.json',
                NORMAL_RUN/'summary.json', MODEL_PATHS['B_rotate10'].parent/'summary.json',
                ROTATION_RUN/'test_manifest.json', NORMAL_RUN/'test_manifest.json',
                BASELINE_DIR/'comparison.csv', BASELINE_DIR/'image_scores.csv']
source_hashes = {p.relative_to(ROOT).as_posix():sha256(p) for p in source_files}
baseline_comparison = pd.read_csv(BASELINE_DIR/'comparison.csv').set_index('condition')
baseline_scores = pd.read_csv(BASELINE_DIR/'image_scores.csv').set_index('relative_path')
assert np.isclose(baseline_comparison.loc['A_original','auroc'],0.82625,atol=1e-12,rtol=0)
assert np.isclose(baseline_comparison.loc['B_rotate10','auroc'],0.8066666666666666,atol=1e-12,rtol=0)


known_config = read_json(KNOWN_DIR/'config.json')
assert known_config['status'] == 'completed'
assert known_config['model_hashes'] == EXPECTED_HASHES
assert known_config['allowed_angles'] == list(range(36,360,36))
assert known_config['assignment_seed'] == 42
assert known_config['ssim_window'] == 11 and known_config['top_k'] == TOP_K
assert known_config['data_range'] == 1
known_scores = pd.read_csv(KNOWN_DIR/'image_scores.csv')
known_assignment = pd.read_csv(KNOWN_DIR/'angle_assignment.csv')
known_comparison = pd.read_csv(KNOWN_DIR/'scenario_comparison.csv')
for filename in ['config.json','image_scores.csv','angle_assignment.csv','scenario_comparison.csv','test_manifest.json']:
    path = KNOWN_DIR/filename
    source_hashes[path.relative_to(ROOT).as_posix()] = sha256(path)


In [ ]:
# 全条件で画像全体を使う。中央820の切り出しは行わない。
final_transform=transforms.Compose([
    transforms.Resize((128,128),interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.Grayscale(num_output_channels=1),
])
to_tensor=transforms.ToTensor()
def rotation_pad_width(size):
    return math.ceil((math.hypot(*size)-min(size))/2)+4

def rotate_with_reflect(image,angle,verify_support=False):
    if angle==0: return image.copy()
    width,height=image.size
    pad=rotation_pad_width(image.size)
    pixels=np.asarray(image)
    extended=Image.fromarray(np.pad(pixels,((pad,pad),(pad,pad),(0,0)),mode='reflect'))
    box=(pad,pad,pad+width,pad+height)
    options=dict(interpolation=transforms.InterpolationMode.NEAREST,expand=False)
    out=TF.rotate(extended,angle=angle,fill=255,**options).crop(box)
    mask=TF.rotate(Image.new('L',image.size,255),angle=angle,fill=0,**options)
    original_part=TF.rotate(image,angle=angle,fill=255,**options)
    valid=np.asarray(mask)>0
    result=np.asarray(out).copy()
    result[valid]=np.asarray(original_part)[valid]
    if verify_support:
        support=TF.rotate(Image.new('L',extended.size,255),angle=angle,fill=0,**options).crop(box)
        assert np.all(np.asarray(support)==255),'パディングが不足しています。'
        assert np.array_equal(result[valid],np.asarray(original_part)[valid])
    return Image.fromarray(result)


In [ ]:
TEST_DIR = ROOT/'data/transistor/test'
expected_manifest = read_json(ROTATION_RUN/'test_manifest.json')
assert expected_manifest == read_json(NORMAL_RUN/'test_manifest.json')
paths = sorted(TEST_DIR.glob('*/*.png'),key=lambda p:p.relative_to(TEST_DIR).as_posix())
assert [p.relative_to(ROOT).as_posix() for p in paths] == [r['repo_path'] for r in expected_manifest]
base_images,base_tensors,image_records = [],[],[]
for path,record in zip(paths,expected_manifest):
    raw = path.read_bytes()
    assert hashlib.sha256(raw).hexdigest() == record['file_sha256']
    with Image.open(BytesIO(raw)) as im:
        assert im.size == (1024,1024)
        image = im.convert('RGB').copy()
    tensor = to_tensor(final_transform(image))
    assert hashlib.sha256(tensor.numpy().tobytes()).hexdigest() == record['preprocessed_sha256']
    base_images.append(image);base_tensors.append(tensor);image_records.append(record)
base_tensor = torch.stack(base_tensors)
image_table = pd.DataFrame(image_records)
counts = image_table.category.value_counts().to_dict()
assert counts == {'good':60,'bent_lead':10,'cut_lead':10,'damaged_case':10,'misplaced':10}
manifest_hash = hashlib.sha256(json.dumps(image_records,ensure_ascii=False,sort_keys=True).encode('utf-8')).hexdigest()
assert manifest_hash == normal_config['test_manifest_sha256'] == rotation_config['test_manifest_sha256']
assert np.array_equal(baseline_scores.loc[image_table.relative_path,'true_label'],image_table.true_label)
display(pd.DataFrame([{'種類':CATEGORY_LABELS[k],'枚数':counts[k]} for k in CATEGORY_LABELS]))


assert read_json(KNOWN_DIR/'test_manifest.json') == image_records
assert known_config['test_manifest_sha256'] == manifest_hash
assert not known_assignment.relative_path.duplicated().any()
known_assignment = known_assignment.set_index('relative_path').loc[image_table.relative_path].reset_index()
assert len(known_assignment) == 100
assert np.array_equal(known_assignment.true_label, image_table.true_label)
assert np.isin(known_assignment.assigned_angle, range(36,360,36)).all()
for name in CONDITIONS:
    part = known_scores[known_scores.condition == name].set_index('relative_path').loc[image_table.relative_path]
    assert len(part) == 100 and part.index.is_unique
    assert np.array_equal(part.true_label, image_table.true_label)
    assert np.array_equal(part.category, image_table.category)
    assert np.array_equal(part.angle, known_assignment.assigned_angle)


## 2. 評価前に全角度を固定する

画像はファイル名順。画像の正解種類・スコア・モデルの出力を見ずに抽選します。
条件ごと・シードごとに再現可能な乱数列を用意し、同じ割り当てを両モデルで共用します。
角度ごとの枚数は均等にそろえず、各候補から独立・等確率・重複ありで抽選します。
中間角度の学習角度からの距離は18度、未知ランダムでは1〜18度です。


In [ ]:
assert ANGLE_CANDIDATES['midpoint'] == tuple(range(18,360,36))
assert len(ANGLE_CANDIDATES['unseen_random']) == 350
assert len(set(ASSIGNMENT_SEEDS)) == 5
assert REPRESENTATIVE_SEED in ASSIGNMENT_SEEDS and set(VISUAL_SEEDS).issubset(ASSIGNMENT_SEEDS)
assert set(VISUAL_SCENARIOS).issubset(ANGLE_CANDIDATES)
assignments = {}
for scenario, candidates in ANGLE_CANDIDATES.items():
    assert all(a != 0 and a % 36 != 0 for a in candidates)
    for seed in ASSIGNMENT_SEEDS:
        # シード＋条件番号で乱数列を独立に固定する（Pythonのhashは使わない）。
        stream = np.random.SeedSequence([int(seed), list(ANGLE_CANDIDATES).index(scenario)])
        rng = np.random.default_rng(stream)
        angles = rng.choice(np.array(candidates,dtype=int), size=len(image_table),replace=True)
        assignment = image_table[['relative_path','category','category_ja','true_label']].copy()
        assignment['scenario'] = scenario
        assignment['assignment_seed'] = int(seed)
        assignment['assigned_angle'] = angles
        assignment['distance_to_training_angle'] = np.minimum(angles % 36,36 - angles % 36)
        assert len(assignment) == assignment.relative_path.nunique() == 100
        assert np.isin(angles,candidates).all() and (angles % 36 != 0).all()
        assignments[(scenario,seed)] = assignment
all_assignments = pd.concat(assignments.values(),ignore_index=True)
assert len(all_assignments) == 1000
display(all_assignments.groupby(['scenario','assignment_seed']).agg(
    画像数=('relative_path','size'),最小角度=('assigned_angle','min'),最大角度=('assigned_angle','max'),
    平均距離=('distance_to_training_angle','mean')))
for scenario in ANGLE_CANDIDATES:
    display(assignments[(scenario,REPRESENTATIVE_SEED)])

if VISUAL_PATHS is not None:
    assert len(VISUAL_PATHS) == len(set(VISUAL_PATHS))
    assert set(VISUAL_PATHS).issubset(set(image_table.relative_path))
    visual_indices = image_table.index[image_table.relative_path.isin(VISUAL_PATHS)].tolist()
else:
    assert MAX_IMAGES_PER_CATEGORY is None or (isinstance(MAX_IMAGES_PER_CATEGORY,int)
        and not isinstance(MAX_IMAGES_PER_CATEGORY,bool) and MAX_IMAGES_PER_CATEGORY > 0)
    visual_indices = []
    for category in CATEGORY_LABELS:
        indices = image_table.index[image_table.category == category].tolist()
        visual_indices.extend(indices if MAX_IMAGES_PER_CATEGORY is None else indices[:MAX_IMAGES_PER_CATEGORY])
print(f'可視化対象は{len(visual_indices)}画像。評価枚数は各回100枚で変わりません。')


## 3. 前回と同じモデル・平均μによる復元・SSIM Top-130

モデル定義と回転関数は前回コードをそのまま引き継ぎます。学習処理はありません。
SSIMは窓11・data_range=1、誤差マップの全画素から上位130点を平均。
AUROCは同点スコアを考慮し、順位による計算との一致もユーザー実行時に照合します。


In [ ]:
class VAE_Hybrid_CNN_FC(nn.Module):
    def __init__(self, latent_dim=64):
        super(VAE_Hybrid_CNN_FC, self).__init__()

        # -------------------------------------------------------------
        # 【エンコーダ前半】 CNN_Deep5（空間圧縮: 128x128 -> 4x4）
        # -------------------------------------------------------------
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU()
        )

        # -------------------------------------------------------------
        # 【エンコーダ後半】 FC_Only（段階的圧縮: 4096次元 -> 128次元）
        # -------------------------------------------------------------
        self.encoder_fc = nn.Sequential(
            nn.Linear(256 * 4 * 4, 1024), nn.ReLU(),
            nn.Linear(1024, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU()
        )

        # 潜在空間へのマッピング（128次元 -> 64次元）
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        # -------------------------------------------------------------
        # 【デコーダ前半】 FC_Only（段階的復元: 64次元 -> 4096次元）
        # -------------------------------------------------------------
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 1024), nn.ReLU(),
            # ※CNNへ繋ぐため、最後はSigmoidではなくReLUを使用します
            nn.Linear(1024, 256 * 4 * 4), nn.ReLU()
        )

        # -------------------------------------------------------------
        # 【デコーダ後半】 CNN_Deep5（空間復元: 4x4 -> 128x128）
        # -------------------------------------------------------------
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),

            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),

            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x, sample=True):
        # 1. CNNによる特徴抽出
        h = self.encoder_conv(x)
        h = torch.flatten(h, start_dim=1)

        # 2. FCによる段階的圧縮
        h = self.encoder_fc(h)

        # 3. 潜在変数の計算と安全装置（クランプ）
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)

        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std if sample else mu

        # 4. FCによる段階的復元
        h_recon = self.decoder_fc(z)
        h_recon = h_recon.view(-1, 256, 4, 4) # 4階テンソルに戻す

        # 5. CNNによる画像復元
        recon = self.decoder_conv(h_recon)

        return recon, mu, logvar


In [ ]:
def binary_roc_curve(y_true, scores):
    labels = np.asarray(y_true)
    scores = np.asarray(scores, dtype=float)
    if labels.ndim != 1 or scores.ndim != 1 or labels.shape != scores.shape:
        raise ValueError('ラベルとスコアは同じ長さの1次元配列にしてください。')
    if not np.isfinite(scores).all() or not np.isin(labels, [0, 1]).all():
        raise ValueError('スコアは有限値、ラベルは0または1である必要があります。')
    positives = int((labels == 1).sum())
    negatives = int((labels == 0).sum())
    if not positives or not negatives:
        raise ValueError('正常と異常の両方が必要です。')
    order = np.argsort(-scores, kind='stable')
    ordered_scores = scores[order]
    ordered_labels = labels[order]
    ends = np.r_[np.flatnonzero(np.diff(ordered_scores)), len(scores) - 1]
    true_positives = np.cumsum(ordered_labels)[ends]
    false_positives = (ends + 1) - true_positives
    fpr = np.r_[0.0, false_positives / negatives]
    tpr = np.r_[0.0, true_positives / positives]
    thresholds = np.r_[np.inf, ordered_scores[ends]]
    return fpr, tpr, thresholds

def binary_auroc(y_true, scores):
    fpr, tpr, _ = binary_roc_curve(y_true, scores)
    return float(np.trapezoid(tpr, fpr))

assert binary_auroc([0,0,1,1], [0.1,0.2,0.8,0.9]) == 1.0
assert binary_auroc([0,0,1,1], [0.9,0.8,0.2,0.1]) == 0.0
assert binary_auroc([0,0,1,1], [0.5,0.5,0.5,0.5]) == 0.5



def verify_auroc(y,scores):
    value=binary_auroc(y,scores)
    normal=scores[y==0]; abnormal=scores[y==1]
    rank=((abnormal[:,None]>normal).sum()+0.5*(abnormal[:,None]==normal).sum())/(len(normal)*len(abnormal))
    assert np.isclose(value,rank,atol=1e-12)
    return value


In [ ]:
def score_loaded_model(model,images):
    model.eval()
    reconstructed = np.empty((len(images),128,128),dtype=np.float32)
    error_maps = np.empty_like(reconstructed)
    scores = np.empty(len(images),dtype=np.float64)
    with torch.inference_mode():
        for start in range(0,len(images),EVAL_BATCH_SIZE):
            x = images[start:start+EVAL_BATCH_SIZE].to(DEVICE)
            output,_,_ = model(x,sample=False)
            assert torch.isfinite(output).all()
            for local in range(len(x)):
                index = start+local
                a = x[local,0].cpu().numpy()
                b = output[local,0].cpu().numpy()
                _,ssim_map = ssim(a,b,data_range=1.0,win_size=11,full=True)
                error = 1.0-ssim_map
                assert np.isfinite(error).all()
                reconstructed[index] = b
                error_maps[index] = error
                scores[index] = np.sort(error.ravel())[::-1][:TOP_K].mean()
    return scores,reconstructed,error_maps


## 4. 回転なし・既知角度の保存済み結果を読み込む

再推論しません。既知角度は前回のseed42による1通りの割り当てなので、5回の平均と同等の確かさとは扱いません。
両方とも同じ元テスト100枚の結果であること、ラベル・モデル・AUROCを確認します。


In [ ]:
baseline_parts, baseline_rows = {}, []
for name in CONDITIONS:
    unrotated = image_table[['relative_path','category','category_ja','true_label']].copy()
    unrotated['score'] = baseline_scores.loc[image_table.relative_path,name].to_numpy()
    unrotated['angle'] = 0
    known = known_scores[known_scores.condition == name].set_index('relative_path').loc[image_table.relative_path].reset_index()
    for scenario,part in [('unrotated',unrotated),('known_rotated',known)]:
        auc = verify_auroc(part.true_label.to_numpy(),part.score.to_numpy())
        if scenario == 'unrotated':
            expected_auc = baseline_comparison.loc[name,'auroc']
        else:
            saved = known_comparison[(known_comparison.condition == name) & (known_comparison.scenario == 'random_rotated')]
            assert len(saved) == 1
            expected_auc = saved.iloc[0].auroc
        assert np.isclose(auc,expected_auc,rtol=0,atol=1e-12)
        baseline_parts[(name,scenario)] = part
        baseline_rows.append({'condition':name,'scenario':scenario,'auroc':auc,'sample_count':100,
            'assignment_count':1,'source':'saved_scores_no_inference'})
baseline_table = pd.DataFrame(baseline_rows)
display(baseline_table.replace({'condition':LABELS,'scenario':SCENARIO_LABELS}))


## 5. 新しい10ケースを評価する（学習はしない）

このセルで新しい実行フォルダを作成し、各ケースの回転画像を一度作り、2モデルで共用します。
評価時の入力・復元・誤差マップをケース別の圧縮NPZに保存します。後の表示では再推論しません。
途中で停止した場合は未完了として記録されます。再実行時は新規フォルダで最初から評価します。


In [ ]:
run_id = datetime.now(ZoneInfo('Asia/Tokyo')).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
RESULT_DIR = OUTPUT_BASE/run_id
RESULT_DIR.mkdir(parents=True,exist_ok=False)
(RESULT_DIR/'cases').mkdir()
settings = {'experiment':'transistor_unseen_rotation_20261117','run_id':run_id,'status':'running',
    'created_at_jst':datetime.now(ZoneInfo('Asia/Tokyo')).isoformat(),
    'models':{k:p.relative_to(ROOT).as_posix() for k,p in MODEL_PATHS.items()},
    'model_hashes':EXPECTED_HASHES,'source_hashes':source_hashes,'test_manifest_sha256':manifest_hash,
    'normal_training_config':normal_config,'rotation_training_config':rotation_config,
    'angle_candidates':{k:list(v) for k,v in ANGLE_CANDIDATES.items()},
    'assignment_seeds':list(ASSIGNMENT_SEEDS),'random_stream':'numpy SeedSequence([seed, scenario_index])',
    'assignment_method':'independent uniform choice with replacement, no label or score conditioning',
    'training_angles':list(TRAINING_ANGLES),'sample_count_per_case':100,'source_count':100,
    'preprocessing':'Full RGB1024 -> reflect pad -> NEAREST rotation -> central1024 -> preserve original-supported pixels -> LANCZOS128 -> grayscale -> ToTensor',
    'rotation_padding_width':rotation_pad_width((1024,1024)),
    'inference':'posterior mean, model.eval, no sampling','top_k':TOP_K,'ssim_window':11,'data_range':1,
    'device':str(DEVICE),'batch_size':EVAL_BATCH_SIZE,'torch_threads':torch.get_num_threads(),
    'baseline_source':BASELINE_DIR.relative_to(ROOT).as_posix(),
    'known_angle_source':KNOWN_DIR.relative_to(ROOT).as_posix(),'baseline_inference_performed':False,
    'representative_seed':REPRESENTATIVE_SEED,'python':platform.python_version(),
    'versions':{v:importlib.metadata.version(v) for v in ['torch','torchvision','numpy','pandas','Pillow','scikit-image']},
    'label_assumption':'Synthetic global rotation changes camera orientation; original labels retained; misplaced requires caution.'}
write_json(RESULT_DIR/'config.json',settings)
write_json(RESULT_DIR/'test_manifest.json',image_records)
all_assignments.to_csv(RESULT_DIR/'angle_assignments.csv',index=False)
baseline_table.to_csv(RESULT_DIR/'saved_baseline_auroc.csv',index=False)
np.savez_compressed(RESULT_DIR/'original_inputs.npz',inputs=base_tensor.numpy())
score_rows, input_records, trial_rows = [], [], []
models = {}
try:
    for name,path in MODEL_PATHS.items():
        model = VAE_Hybrid_CNN_FC(latent_dim=64).to(DEVICE)
        model.load_state_dict(torch.load(path,map_location=DEVICE,weights_only=True),strict=True)
        model.eval()
        model.requires_grad_(False)
        models[name] = model
    for (scenario,seed),assignment in assignments.items():
        case_id = f'{scenario}_seed{seed}'
        case_dir = RESULT_DIR/'cases'/case_id
        case_dir.mkdir(exist_ok=False)
        rotated_tensors = []
        for index,(image,angle) in enumerate(zip(base_images,assignment.assigned_angle)):
            angle = int(angle)
            transformed = to_tensor(final_transform(rotate_with_reflect(image,angle,verify_support=True)))
            rotated_tensors.append(transformed)
            input_records.append({'scenario':scenario,'assignment_seed':int(seed),
                'relative_path':image_table.loc[index,'relative_path'],'angle':angle,
                'source_sha256':image_records[index]['file_sha256'],
                'preprocessed_sha256':hashlib.sha256(transformed.numpy().tobytes()).hexdigest()})
        rotated_tensor = torch.stack(rotated_tensors)
        assert tuple(rotated_tensor.shape) == (100,1,128,128)
        assert torch.isfinite(rotated_tensor).all() and 0 <= rotated_tensor.min() <= rotated_tensor.max() <= 1
        arrays = {'rotated_inputs':rotated_tensor.numpy(),
            'angles':assignment.assigned_angle.to_numpy(dtype=np.int16),
            'relative_paths':image_table.relative_path.to_numpy(dtype=str)}
        for name in CONDITIONS:
            scores,reconstructed,maps = score_loaded_model(models[name],rotated_tensor)
            auc = verify_auroc(image_table.true_label.to_numpy(),scores)
            arrays[f'{name}_reconstruction'] = reconstructed
            arrays[f'{name}_error_map'] = maps
            arrays[f'{name}_scores'] = scores
            for index,row in image_table.iterrows():
                score_rows.append({'scenario':scenario,'assignment_seed':int(seed),'condition':name,
                    'relative_path':row.relative_path,'category':row.category,'category_ja':row.category_ja,
                    'true_label':int(row.true_label),'angle':int(assignment.loc[index,'assigned_angle']),
                    'score':float(scores[index])})
            trial_rows.append({'scenario':scenario,'assignment_seed':int(seed),'condition':name,
                'sample_count':100,'source_count':100,'normal_count':60,'anomaly_count':40,'auroc':auc})
            print(f'{SCENARIO_LABELS[scenario]} / seed{seed} / {LABELS[name]} : AUROC={auc:.5f}',flush=True)
        np.savez_compressed(case_dir/'evaluation_arrays.npz',**arrays)
        pd.DataFrame(score_rows).to_csv(RESULT_DIR/'image_scores.csv',index=False)
        pd.DataFrame(trial_rows).to_csv(RESULT_DIR/'trial_auroc.csv',index=False)
        write_json(RESULT_DIR/'rotated_input_manifest.json',input_records)
        del rotated_tensors,rotated_tensor,arrays,scores,reconstructed,maps
    scores_table = pd.DataFrame(score_rows)
    trial_auroc = pd.DataFrame(trial_rows)
    assert len(scores_table) == 2000 and len(trial_auroc) == 20
    assert not scores_table.duplicated(['scenario','assignment_seed','condition','relative_path']).any()
    for (scenario,seed,name),part in scores_table.groupby(['scenario','assignment_seed','condition']):
        aligned = part.set_index('relative_path').loc[image_table.relative_path]
        assert len(aligned) == 100 and aligned.category.value_counts().to_dict() == counts
        assert np.array_equal(aligned.angle,assignments[(scenario,seed)].assigned_angle)
    settings.update(status='inference_completed',completed_cases=len(assignments))
    write_json(RESULT_DIR/'config.json',settings)
except BaseException as exc:
    settings.update(status='interrupted_or_failed',error=f'{type(exc).__name__}: {exc}')
    write_json(RESULT_DIR/'config.json',settings)
    raise
finally:
    models.clear()
    if 'model' in globals(): del model
print('保存先：',RESULT_DIR)


## 6. 各回・平均・標準偏差・モデル間の差

各100枚で算出した5つのAUROCを平均します。スコアを500枚分まとめたAUROCではありません。
標準偏差は標本標準偏差（ddof=1）。誤差棒は信頼区間ではありません。
モデル間の差は、同じ角度割り当てでの「回転拡張−通常学習」を先に計算します。


In [ ]:
summary = trial_auroc.groupby(['scenario','condition'],sort=False).agg(
    auroc_mean=('auroc','mean'),auroc_std=('auroc','std'),auroc_min=('auroc','min'),
    auroc_max=('auroc','max'),assignment_count=('auroc','size')).reset_index()
assert (summary.assignment_count == 5).all()
paired = trial_auroc.pivot(index=['scenario','assignment_seed'],columns='condition',values='auroc')
paired['delta_rotation_minus_normal'] = paired['B_rotate10']-paired['A_original']
delta_summary = paired.groupby(level='scenario').delta_rotation_minus_normal.agg(['mean','std','min','max'])
display(trial_auroc.replace({'scenario':SCENARIO_LABELS,'condition':LABELS}))
display(summary.replace({'scenario':SCENARIO_LABELS,'condition':LABELS}))
display(paired.rename(columns=LABELS)); display(delta_summary)
summary.to_csv(RESULT_DIR/'auroc_summary.csv',index=False)
paired.to_csv(RESULT_DIR/'paired_model_differences.csv')
delta_summary.to_csv(RESULT_DIR/'paired_difference_summary.csv')

comparison_rows = []
for row in baseline_rows:
    comparison_rows.append({**row,'auroc_mean':row['auroc'],'auroc_std':np.nan})
for row in summary.to_dict('records'):
    comparison_rows.append({**row,'sample_count':100,'source':'new_evaluation'})
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(RESULT_DIR/'scenario_comparison.csv',index=False)
display(comparison.replace({'scenario':SCENARIO_LABELS,'condition':LABELS}))
fig,ax = plt.subplots(figsize=(12,6),constrained_layout=True)
x = np.arange(len(SCENARIO_LABELS)); width=.34
for offset,name in enumerate(CONDITIONS):
    part = comparison[comparison.condition == name].set_index('scenario').loc[list(SCENARIO_LABELS)]
    positions = x+(offset-.5)*width
    ax.bar(positions,part.auroc_mean,width,label=LABELS[name],color=COLORS[name],alpha=.85)
    mask = part.auroc_std.notna().to_numpy()
    ax.errorbar(positions[mask],part.auroc_mean.to_numpy()[mask],
        yerr=part.auroc_std.to_numpy()[mask],fmt='none',ecolor='black',capsize=5)
    for px,value in zip(positions,part.auroc_mean):
        ax.text(px,value+.035,f'{value:.4f}',ha='center',fontsize=10)
ax.set(xticks=x,xticklabels=list(SCENARIO_LABELS.values()),ylabel='AUROC',ylim=(0,1.12),
    title='transistor：回転条件による比較（未知角度は平均±標準偏差、各回100枚）')
ax.grid(axis='y',ls='--',alpha=.5);ax.set_axisbelow(True);ax.legend()
fig.savefig(RESULT_DIR/'scenario_auroc.png',dpi=180);plt.show();plt.close(fig)
fig,axes = plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for ax,scenario in zip(axes,ANGLE_CANDIDATES):
    for name in CONDITIONS:
        part = trial_auroc[(trial_auroc.scenario == scenario)&(trial_auroc.condition == name)]
        ax.plot(part.assignment_seed,part.auroc,'o-',color=COLORS[name],label=LABELS[name])
    ax.set(title=SCENARIO_LABELS[scenario],xlabel='角度割り当てseed（学習seedではない）',ylabel='AUROC',ylim=(0,1.05),xticks=ASSIGNMENT_SEEDS)
    ax.grid(ls='--',alpha=.5);ax.legend()
fig.savefig(RESULT_DIR/'auroc_by_assignment.png',dpi=180);plt.show();plt.close(fig)


## 7. 代表回（初期seed42）のROCとスコア分布

モデルごとの性能を見て代表回を選びません。ROCは8×8・右下の凡例・破線グリッド・対角線なし。
ヒストグラムは10×6、正常＝緑／異常＝赤、40点の共通ビン境界を使います。判定閾値は設定しません。


In [ ]:
for scenario in ANGLE_CANDIDATES:
    selected = scores_table[(scores_table.scenario==scenario)&(scores_table.assignment_seed==REPRESENTATIVE_SEED)]
    fig,ax = plt.subplots(figsize=(8,8),constrained_layout=True)
    for name in CONDITIONS:
        part = selected[selected.condition==name]
        fpr,tpr,_ = binary_roc_curve(part.true_label.to_numpy(),part.score.to_numpy())
        auc = verify_auroc(part.true_label.to_numpy(),part.score.to_numpy())
        ax.plot(fpr,tpr,color=COLORS[name],lw=2,label=f'{LABELS[name]} (AUROC={auc:.4f})')
    ax.set(xlim=(0,1),ylim=(0,1.05),xlabel='False Positive Rate（正常を誤検出した割合）',
        ylabel='True Positive Rate（異常を検出した割合）',title=f'ROC：{SCENARIO_LABELS[scenario]} / seed{REPRESENTATIVE_SEED}')
    ax.legend(loc='lower right');ax.grid(ls='--',alpha=.6)
    fig.savefig(RESULT_DIR/f'roc_{scenario}_seed{REPRESENTATIVE_SEED}.png',dpi=180);plt.show();plt.close(fig)
    lo,hi = float(selected.score.min()),float(selected.score.max())
    if lo == hi: lo-=1e-6;hi+=1e-6
    bins = np.linspace(lo,hi,40)
    ymax = max(np.histogram(selected.loc[(selected.condition==name)&(selected.true_label==label),'score'],bins=bins)[0].max()
        for name in CONDITIONS for label in [0,1])+1
    for name in CONDITIONS:
        part = selected[selected.condition==name]
        fig,ax = plt.subplots(figsize=(10,6),constrained_layout=True)
        for label,color,label_text in [(0,'green','正常'),(1,'red','異常')]:
            ax.hist(part.loc[part.true_label==label,'score'],bins=bins,color=color,alpha=.5,edgecolor='white',label=label_text)
        ax.set(xlabel='異常スコア（SSIM誤差上位130点の平均）',ylabel='画像枚数',xlim=(lo,hi),ylim=(0,ymax),
            title=f'{SCENARIO_LABELS[scenario]} / seed{REPRESENTATIVE_SEED} / {LABELS[name]}')
        ax.legend();ax.grid(axis='y',ls='--',alpha=.5)
        fig.savefig(RESULT_DIR/f'hist_{scenario}_seed{REPRESENTATIVE_SEED}_{name}.png',dpi=180);plt.show();plt.close(fig)


## 8. 異常種類別AUROC（正常60枚と各異常10枚）

全体AUROCだけでなく、配置ずれなどの種類別に確認します。
各異常は10枚と少数です。種類別AUROCの平均は全体AUROCとは別の値です。


In [ ]:
category_rows = []
def append_category_metrics(name,scenario,seed,part,source):
    for category in CATEGORY_LABELS:
        if category == 'good': continue
        subset = part[part.category.isin(['good',category])]
        assert len(subset)==70 and (subset.true_label==0).sum()==60 and (subset.true_label==1).sum()==10
        category_rows.append({'condition':name,'scenario':scenario,'assignment_seed':seed,
            'category':category,'category_ja':CATEGORY_LABELS[category],'source':source,
            'normal_count':60,'anomaly_count':10,
            'auroc':verify_auroc(subset.true_label.to_numpy(),subset.score.to_numpy())})
for (name,scenario),part in baseline_parts.items():
    append_category_metrics(name,scenario,42 if scenario=='known_rotated' else -1,part,'saved')
for (scenario,seed,name),part in scores_table.groupby(['scenario','assignment_seed','condition']):
    append_category_metrics(name,scenario,seed,part,'new')
category_auroc = pd.DataFrame(category_rows)
category_summary = category_auroc.groupby(['scenario','condition','category','category_ja'],sort=False).agg(
    auroc_mean=('auroc','mean'),auroc_std=('auroc','std'),assignment_count=('auroc','size')).reset_index()
display(category_summary.replace({'condition':LABELS,'scenario':SCENARIO_LABELS}))
category_auroc.to_csv(RESULT_DIR/'category_trial_auroc.csv',index=False)
category_summary.to_csv(RESULT_DIR/'category_auroc_summary.csv',index=False)
for scenario in ANGLE_CANDIDATES:
    categories = [c for c in CATEGORY_LABELS if c!='good']
    fig,ax = plt.subplots(figsize=(11,6),constrained_layout=True)
    x = np.arange(len(categories))
    for offset,name in enumerate(CONDITIONS):
        part = category_summary[(category_summary.scenario==scenario)&(category_summary.condition==name)].set_index('category').loc[categories]
        ax.bar(x+(offset-.5)*.34,part.auroc_mean,width=.34,yerr=part.auroc_std,capsize=4,
            color=COLORS[name],label=LABELS[name])
    ax.set(xticks=x,xticklabels=[CATEGORY_LABELS[c] for c in categories],ylim=(0,1.12),ylabel='AUROC',
        title=f'異常種類別：{SCENARIO_LABELS[scenario]} / 平均±標準偏差')
    ax.grid(axis='y',ls='--',alpha=.5);ax.set_axisbelow(True);ax.legend()
    fig.savefig(RESULT_DIR/f'category_{scenario}.png',dpi=180);plt.show();plt.close(fig)


## 9. 評価時の入力・復元・誤差マップを表示する

設定セルの `VISUAL_SCENARIOS`・`VISUAL_SEEDS`・`VISUAL_PATHS`・`MAX_IMAGES_PER_CATEGORY` で変更できます。
全100枚の表示は `VISUAL_PATHS=None`、`MAX_IMAGES_PER_CATEGORY=None`。
設定を変えた場合は「2. 評価前に全角度を固定する」のセルも実行し、表示画像の選択を更新してください。
その後はこのセルだけで保存配列から再表示できます。**評価セルの再実行は不要**です。
画像は0〜1、誤差マップは0〜2で全図共通。色は1−SSIMであり欠陥の正解位置ではありません。


In [ ]:
visual_dir = RESULT_DIR/'visualizations'
visual_dir.mkdir(exist_ok=True)
for scenario in VISUAL_SCENARIOS:
    for seed in VISUAL_SEEDS:
        assert scenario in ANGLE_CANDIDATES and seed in ASSIGNMENT_SEEDS
        case_path = RESULT_DIR/'cases'/f'{scenario}_seed{seed}'/'evaluation_arrays.npz'
        with np.load(case_path,allow_pickle=False) as stored:
            # 圧縮配列をケースごとに一度だけ展開し、全画像表示時の再読み込みを避ける。
            arrays = {key:stored[key] for key in stored.files}
            assert np.array_equal(arrays['relative_paths'],image_table.relative_path.to_numpy(dtype=str))
            assignment = assignments[(scenario,seed)]
            assert np.array_equal(arrays['angles'],assignment.assigned_angle)
            for name in CONDITIONS:
                part = scores_table[(scores_table.scenario==scenario)&(scores_table.assignment_seed==seed)&(scores_table.condition==name)]
                saved_scores = part.set_index('relative_path').loc[image_table.relative_path,'score'].to_numpy()
                assert np.allclose(arrays[f'{name}_scores'],saved_scores,rtol=0,atol=1e-12)
            for index in visual_indices:
                record = image_table.loc[index]; angle = int(arrays['angles'][index])
                fig,axes = plt.subplots(2,4,figsize=(13,7),constrained_layout=True)
                for row,name in enumerate(CONDITIONS):
                    axes[row,0].imshow(base_tensor[index,0],cmap='gray',vmin=0,vmax=1,interpolation='nearest')
                    axes[row,1].imshow(arrays['rotated_inputs'][index,0],cmap='gray',vmin=0,vmax=1,interpolation='nearest')
                    axes[row,2].imshow(arrays[f'{name}_reconstruction'][index],cmap='gray',vmin=0,vmax=1,interpolation='nearest')
                    heat=axes[row,3].imshow(arrays[f'{name}_error_map'][index],cmap='inferno',vmin=0,vmax=2,interpolation='nearest')
                    axes[row,0].set_ylabel(LABELS[name])
                    score=float(arrays[f'{name}_scores'][index])
                    for col,title in enumerate(['回転前の入力',f'割り当て：{angle}度',f'復元 / スコア={score:.4f}','SSIM誤差マップ']):
                        axes[row,col].set_title(title,fontsize=11);axes[row,col].set_xticks([]);axes[row,col].set_yticks([])
                fig.colorbar(heat,ax=axes[:,3],label='1−SSIM（全図共通）',shrink=.8)
                fig.suptitle(f'{record.relative_path} / 元の正解：{record.category_ja}\n{SCENARIO_LABELS[scenario]} / seed{seed} / 平均μから復元',fontsize=13)
                stem=record.relative_path.replace('/','_').replace('.png','')
                fig.savefig(visual_dir/f'{scenario}_seed{seed}_{stem}_{angle:03d}deg.png',dpi=160)
                plt.show();plt.close(fig)


## 10. 完了記録と解釈の注意点

- 中間角度でも回転拡張モデルの性能が高ければ、固定した学習角度の間への対応を示唆します。
- 2つの角度分布は異なります。中間角度よりランダム角度で高性能でも「ランダム回転の方が難しい／簡単」と一律には言えません。
- 学習seedは1つ。5回の割り当てで学習の安定性や統計的優位性が証明されるわけではありません。
- 画像全体の回転を撮影方向の変化と仮定し、元の正常／異常ラベルを維持しています。
  特にmisplacedの配置異常と、検査で許容する向きの変化を区別して種類別結果を確認してください。
- 反射補完で模様や対象の一部が複製される可能性、固定枠で対象が切れる可能性があります。
  実際に向きを変えて撮影した画像の評価ではなく、未知角度も合成回転への対応です。
- このテストは既に結果を見た後の探索的追試です。結果を見て代表seed・角度・βを選ぶためには使いません。
- **bottleへの効果はまだ結論にしません。** 次段階のランダム回転学習も今回は実装しません。


In [ ]:
assert all(sha256(ROOT/path)==value for path,value in source_hashes.items()), '読み込み元が実行中に変更されました。'
assert all(sha256(ROOT/r['repo_path'])==r['file_sha256'] for r in image_records), '元画像が実行中に変更されました。'
assert len(all_assignments)==1000 and len(scores_table)==2000
assert len(input_records)==1000 and len(trial_auroc)==20
result_files = sorted(p for p in RESULT_DIR.rglob('*') if p.is_file() and p.name not in {'config.json','output_file_hashes.json'})
write_json(RESULT_DIR/'output_file_hashes.json',{p.relative_to(RESULT_DIR).as_posix():sha256(p) for p in result_files})
settings.update(status='completed',completed_at_jst=datetime.now(ZoneInfo('Asia/Tokyo')).isoformat(),
    unique_test_images=100,case_count=10,evaluation_variants=1000,model_score_count=2000,
    baseline_auroc_verified_from_saved_scores=True,
    limitations=['synthetic rotation includes padding/cropping','original labels retained; misplaced requires caution',
        'one training seed; assignment std is not training uncertainty or confidence interval',
        'same 100 source images reused','saved known-angle baseline has one assignment',
        'integer-degree grid; random set includes midpoint angles','exploratory reuse of test dataset'])
write_json(RESULT_DIR/'config.json',settings)
print('評価完了・保存先：',RESULT_DIR)
print('再学習なし。未知角度2条件×5回×100枚×2モデル。bottleは対象外です。')


## 参照・作成時点の確認状態

- 元コード：[261106transistorの検証画像の回転に対する異常検知性能の比較.ipynb](261106transistorの検証画像の回転に対する異常検知性能の比較.ipynb)
- 学習更新数をそろえた比較：[261106transistorの更新回数を揃えた通常学習と回転拡張のAUROC比較.ipynb](261106transistorの更新回数を揃えた通常学習と回転拡張のAUROC比較.ipynb)
- 研究方針：[memory.md 第75節](../../../memory/memory.md)

2026-11-17：コード作成・静的確認のみ。ノートブック・セル・構文テスト・推論・描画はアシスタントが実行していません。
既存のモデル・設定・スコア・元画像は変更していません。実行後の図と結果はユーザーが確認してください。
